In [1]:
import sys
import os

sys.path.insert(0, os.path.abspath("../.."))

In [3]:
from psiqworkbench import Qubits, QPU
from psiqworkbench.integrations import circuit_designer
import numpy as np
from syk_simulation.qubitization.qsp import qsp_evolution

N = 8 
random_seed = 3
time = 1
epsilon = 1e-3
J=1

# Qubit register sizes
system_size = N
index_chunk = int(np.ceil(np.log2(N)))
index_size = 4 * index_chunk
aux_unary_size = index_chunk
range_flag_size = 1
branch_size = 1
mode_size = 1
selection_size = 1

walk_size = branch_size + index_size + system_size
clean_ladder_aux = 8
num_qubits = walk_size + aux_unary_size + range_flag_size + mode_size + selection_size + clean_ladder_aux


# Setup QPU and Qubit registers
# (except for aux_unary and range_flag as they are auxiliary Qubrick qubits)

# qpu = QPU(num_qubits=num_qubits, filters=[">>state-vector-sim>>"])
# qpu = QPU(num_qubits=num_qubits, filters=[">>state-vector-sim>>",">>clean-ladder-filter>>",">>single-control-filter>>"])
# qpu = QPU(num_qubits=num_qubits, filters=[">>state-vector-sim>>",">>clean-ladder-filter>>",">>toffoli-filter>>",">>single-control-filter>>"])

# qpu = QPU(num_qubits=num_qubits, filters=NO_SIM_LONG)

# qpu= QPU(num_qubits=num_qubits, filters=[">>buffer>>"])

# qpu = QPU(num_qubits=100, filters=[">>clean-ladder-filter>>", ">>buffer>>"])
qpu = QPU(num_qubits=num_qubits, filters=[">>toffoli-filter>>", ">>buffer>>"])
# qpu = QPU(num_qubits=num_qubits, filters=[">>single-control-filter>>", ">>buffer>>"])

# qpu = QPU(num_qubits=num_qubits, filters=[">>clean-ladder-filter>>", ">>toffoli-filter>>", ">>buffer>>"]) #">>single-control-filter>>", ">>buffer>>"])
# qpu = QPU(num_qubits=num_qubits, filters=[">>clean-ladder-filter>>", ">>single-control-filter>>"])

# qpu = QPU(num_qubits=num_qubits, filters=[">>clean-ladder-filter>>", ">>toffoli-filter>>",">>single-control-filter>>"])
# qpu = QPU(num_qubits=num_qubits, filters=[">>clean-ladder-filter>>", ">>toffoli-filter>>",">>single-control-filter>>", ">>rs-synth-filter>>"])

walk = Qubits(walk_size, "walk", qpu)
branch = Qubits(walk[0:branch_size], "branch")
index = Qubits(walk[branch_size : index_size + branch_size], "index")
system = Qubits(walk[index_size + branch_size :], "system")
mode = Qubits(mode_size, "mode", qpu=qpu)
selection = Qubits(selection_size, "selection", qpu=qpu)

# PREPROCESSING
# 1) set branch |+>
branch.had()
qsp_evolution(N=N, J=J, branch=branch, index=index, system=system, mode=mode, selection=selection, time=time, epsilon=epsilon, random_seed=random_seed)
branch.had()

circuit_designer.draw(qpu, depth=2)

lambda=4.618802153517006 tau=4.618802153517006
poly_degree=10.9011009691571
[PolyTaylorSeries] (Cheb) max 0.9498001144183129 is at -0.3402244945779234: normalizing
[PolyTaylorSeries] (Cheb) average error = 0.00019738318857966017 in the domain [-1, 1] using degree 10
[sym_qsp] Iterative optimization to err 1.000e-12 or max_iter 100.
iter: 001 --- err: 3.053e-01
iter: 002 --- err: 6.392e-02
iter: 003 --- err: 6.042e-03
iter: 004 --- err: 6.113e-05
iter: 005 --- err: 5.470e-09
iter: 006 --- err: 4.857e-16
[sym_qsp] Stop criteria satisfied.
[PolyTaylorSeries] (Cheb) max 0.95005512693057 is at 0.6801551922927964: normalizing
[PolyTaylorSeries] (Cheb) average error = 3.934452092678828e-05 in the domain [-1, 1] using degree 10
[sym_qsp] Iterative optimization to err 1.000e-12 or max_iter 100.
iter: 001 --- err: 2.184e-01
iter: 002 --- err: 3.916e-02
iter: 003 --- err: 3.310e-03
iter: 004 --- err: 3.483e-05
iter: 005 --- err: 4.570e-09
iter: 006 --- err: 4.489e-16
[sym_qsp] Stop criteria satis

In [3]:
qpu.serialize("my-file.qre-analysis", dialect="basquiat")